In [ ]:
from enum import unique

import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from scipy.sparse import csr_matrix
from scipy.io import mmwrite
from pathlib import Path
import anndata
import copy
from scipy.io import mmread
from scipy.sparse import csr_matrix
import glob
import re
print("scanpy version:", sc.__version__)
print("numpy version:", np.__version__)

In [ ]:
os.getcwd()

In [ ]:
os.chdir('D:\\lxk\\project\\jiangshucai20260506')
os.getcwd()
import helper.plot_helper as hp
hp.setMyStyle()

In [ ]:
import h5py

with h5py.File("./h5ad/sft.h5ad", "r+") as f:
    if "uns/log1p" in f:
        del f["uns/log1p"]
with h5py.File("./h5ad/myeloid.h5ad", "r+") as f:
    if "uns/log1p" in f:
        del f["uns/log1p"]

In [ ]:
CAT = sc.read_h5ad('./h5ad/sft.h5ad')
CAMM = sc.read_h5ad('./h5ad/myeloid.h5ad')
CAMM.obsm['X_umap'][:, 0] += 15

In [ ]:
CAMM = CAMM[CAMM.obs['group'] == 'LT'].copy()

In [ ]:
adata = anndata.concat([CAT,CAMM], join='outer', fill_value=0)

In [ ]:
sc.pl.umap(adata,color='celltype',)

In [ ]:
np.unique(adata.obs['celltype'])

In [ ]:
import omicverse as ov

#ov.plot_set(font_path='Arial')

In [ ]:
cpdb_results, adata_cpdb = ov.single.run_cellphonedb_v5(
    adata,
    cpdb_file_path='./cellphoneDB/cellphonedb.zip',
    celltype_key='celltype',
    min_cell_fraction=0.005,
    min_genes=200,
    min_cells=3,
    iterations=1000,
    threshold=0.1,
    pvalue=0.05,
    threads=10,
    output_dir='./cpdb_results',
    cleanup_temp=True,
)

In [ ]:
adata.uns_keys()

In [ ]:

adata.obs['celltype'] = adata.obs['celltype'].astype('category')

if 'celltype_colors' not in adata.uns:
    fallback_colors = (
        ov.pl.red_color
        + ov.pl.blue_color
        + ov.pl.green_color
        + ov.pl.orange_color
        + ov.pl.purple_color
    )

    adata.uns['celltype_colors'] = fallback_colors[:len(adata.obs['celltype'].cat.categories)]

color_dict = dict(zip(
    adata.obs['celltype'].cat.categories,
    adata.uns['celltype_colors']
))

adata_plot = adata if 'cpdb_results' in adata.uns else adata_cpdb
comm_adata = ov.single.extract_comm_adata(adata, result_uns_key='cpdb_results') if 'cpdb_results' in adata.uns else adata_cpdb

focus_pathway = 'Signaling by Fibroblast growth factor'
focus_pair_lr = 'NCAM1_FGFR1'
focus_ligand = 'FN1'

umap_df = pd.DataFrame(
    adata.obsm['X_umap'][:, :2],
    columns=['x', 'y'],
    index=adata.obs_names,
)
umap_df['cell_type'] = adata.obs['celltype'].astype(str).values
node_positions = umap_df.groupby('cell_type', observed=True)[['x', 'y']].median()
embedding_points = umap_df.reset_index(drop=True)

comm_adata.uns['node_positions'] = node_positions
comm_adata.uns['embedding_points'] = embedding_points
comm_adata.uns['embedding_axes'] = ('UMAP_1', 'UMAP_2')

comparison_comm = comm_adata.copy()
comparison_comm.layers['means'] = np.asarray(comm_adata.layers['means']).copy() * 0.85
comparison_comm.layers['pvalues'] = np.clip(
    np.asarray(comm_adata.layers['pvalues']).copy() * 1.1,
    0.0,
    1.0,
)

adata_plot, comm_adata

In [ ]:
figpath = hp.mydir('./fig/cellphone/LT[(sft_Tumor)_(myeloid)]')

## 总通讯

In [ ]:
fig, ax = ov.pl.ccc_heatmap(
    adata_cpdb,
    plot_type='dot',
    display_by='aggregation',
    cmap='YlGnBu',

    show=False,
)

plt.savefig(f'{figpath}/总的通信.pdf', dpi=500, bbox_inches='tight')

## outgoing

In [ ]:
fig, ax = ov.pl.ccc_network_plot(
    comm_adata,
    plot_type='individual_outgoing',
    palette=color_dict,
    figsize=(20, 20),
    show=False,
)

plt.savefig(f'{figpath}/作为outgoing.pdf', dpi=500, bbox_inches='tight')

## 总的出度 入度

In [ ]:
fig, ax = ov.pl.ccc_stat_plot(
    adata_plot,
    plot_type='scatter',
    figsize=(6, 6),
    show=False,
)
plt.savefig(f'{figpath}/总的出度 入度.pdf', dpi=500, bbox_inches='tight')


## 分析通路

In [ ]:
comm_adata.var['classification'] = comm_adata.var['classification'].fillna('')

In [ ]:
print(np.unique(adata_plot.var['classification']).tolist())

In [ ]:
focused_pathways = [
    "Adhesion by Collagen/Integrin",
    "Adhesion by Fibronectin",
    "Signaling by Fibronectin",
    "Adhesion by Osteopontin",
    "Adhesion by Thrombospondin",
    "Signaling by Chemokines",
    "Signaling by Interleukin",
    "Signaling by Tumor necrosis factor",
    "Signaling by Colony-Stimulating factor",
    "Signaling by Complement",
    "Signaling by HLA",
    "Signaling by Pro-MHC",
    "Signaling by Galectin",
    "Signaling by Transforming growth factor",
    "Signaling by MIF",
]

In [ ]:

for pathway in focused_pathways:

    try:
        tempdir = hp.mydir(f'{figpath}/{pathway}')
        fig, ax = ov.pl.ccc_stat_plot(
            adata_plot,
            plot_type='sankey',
            display_by='interaction',
            signaling=[pathway],
            palette=color_dict,
            top_n=10,
            figsize=(8, 6),
            show=False,
        )

        plt.savefig(f'{tempdir}/1.pdf', dpi=500, bbox_inches='tight')

        fig, ax = ov.pl.ccc_stat_plot(
            adata_plot,
            plot_type='bar',
            figsize=(6, 4),
            top_n=10,
            signaling = pathway,
            show=False,
        )
        plt.savefig(f'{tempdir}/2.pdf', dpi=500, bbox_inches='tight')

        fig, ax = ov.pl.ccc_stat_plot(
            adata_plot,
            plot_type='lr_contribution',
            signaling=pathway,
            figsize=(10, 5),
            show=False,
        )
        plt.savefig(f'{tempdir}/3.pdf', dpi=500, bbox_inches='tight')


        fig, ax = ov.pl.ccc_network_plot(
            comm_adata,
            plot_type='chord',
            signaling=pathway,
            palette=color_dict,
            normalize_to_sender=True,
            figsize=(6, 6),
            show=False,
        )
        plt.savefig(f'{tempdir}/4.pdf', dpi=500, bbox_inches='tight')



        fig, ax = ov.pl.ccc_network_plot(
            comm_adata,
            plot_type='embedding_network',
            signaling=pathway,
            node_positions=node_positions,
            embedding_points=embedding_points,
            palette=color_dict,
            top_n=20,
            figsize=(7, 10),
            show=False,
        )

        plt.savefig(f'{tempdir}/5.pdf', dpi=500, bbox_inches='tight')


        fig, ax = ov.pl.ccc_stat_plot(
            adata_plot,
            plot_type='scatter',
            signaling=pathway,
            figsize=(6, 6),
            show=False,
        )

        plt.savefig(f'{tempdir}/6.pdf', dpi=500, bbox_inches='tight')

        fig, ax = ov.pl.ccc_heatmap(
            adata_plot,
            plot_type='dot',
            sender_use=np.unique(adata.obs['celltype'].tolist()),
            display_by='interaction',
            signaling=pathway,
            top_n=10,
            transpose = True,
            cmap='viridis',
            figsize=(12, 7),
            #add_text = True,
            top_anno='cell',
            left_anno='cell',
            show=False,
        )

        plt.savefig(f'{tempdir}/7.pdf', dpi=500, bbox_inches='tight')

        fig, ax = ov.pl.ccc_heatmap(
            adata_plot,
            plot_type='dot',
            sender_use=np.unique(adata.obs['celltype'].tolist()),
            display_by='interaction',
            signaling=pathway,
            top_n=10,
            transpose = True,
            cmap='viridis',
            figsize=(12, 7),
            #add_text = True,
            top_anno='cell',
            left_anno='cell',
            show=False,
        )

        plt.savefig(f'{tempdir}/8.pdf', dpi=500, bbox_inches='tight')


    except ValueError:
        continue


In [ ]:
np.unique(adata.obs['celltype'])

In [ ]:

fig, ax = ov.pl.ccc_heatmap(
    adata_plot,
    plot_type='dot',
    sender_use=['IFN Mono', 'MAIT', 'TREM2+ DAMs', 'Tcm', 'Teff', 'Tem', 'Tex',
       'Tn', 'Trans Mono', 'Trm', 'cDC2', 'cMono', 'gdT'],
    display_by='interaction',
    signaling=['Signaling by HLA'],
    top_n=10,
    transpose = True,
    cmap='viridis',
    figsize=(12, 7),
    #add_text = True,
    top_anno='cell',
    left_anno='cell',


    show=False,

)
plt.savefig(f'{figpath}/test222222.pdf', dpi=500, bbox_inches='tight')

In [ ]:
adata_plot.var.columns

In [ ]:
adata_plot.var_names

In [ ]:
fig, ax = ov.pl.ccc_heatmap(
    adata_plot,
    plot_type='dot',
    sender_use=['Tn', 'Tcm', 'Teff'],
    display_by='aggregation',
    #signaling=['Adhesion by Cadherin'],
    top_n=5,
    cmap='viridis',
    figsize=(8, 3),
    show=False,
    show_row_names=True,
    show_col_names=True,

)

In [ ]:
fig, ax = ov.pl.ccc_heatmap(
    adata_plot,
    plot_type='focused_heatmap',
    signaling=['Adhesion by CADM'],
    min_interaction_threshold=0.1,
    display_by='interaction',
    cmap='YlGnBu',
    figsize=(7, 5),
    show=False,
    bottom_anno = 'cell'

)

In [ ]:
fig, ax = ov.pl.ccc_network_plot(
    adata_plot,
    plot_type='pathway',
    signaling=['Adhesion by CADM'],
    palette=color_dict,
    top_n=50,
    figsize=(6, 6),
    show=False,
)

In [ ]:
fig, ax = ov.pl.ccc_network_plot(
    comm_adata,
    plot_type='embedding_network',
    signaling=['Adhesion by Collagen/Integrin'],
    node_positions=node_positions,
    embedding_points=embedding_points,
    palette=color_dict,
    top_n=20,
    figsize=(8, 8),
    show=False,
)